# 18. 선수별 95에폭 모델의 원본 영상에서 허브 판독

각 fold에서 학습에 쓰지 않은 선수의 영상만 봅니다. 확인할 질문은 동작 중 시간 연결로 살린 원판이나 놓친 원판이 고신뢰 직접 검출과 어떻게 다른가입니다. 원본을 볼 때는 예측 박스·중심·점수와 짝의 역할을 숨깁니다.

In [ ]:
from wlcv.evaluation.fold_hub_screen import PARENT, scan_originals
from wlcv.evaluation.fold_hub_pairs import prepare_queue
from wlcv.annotation.fold_low_confidence import PAIRED, SPOT, editor, show_after_review

# 기본적으로 이미 완료한 추론 결과를 연다. 파일이 없으면 Mac MPS로 한 번 생성한다.
saved = sorted(PARENT.glob("20*/frames.csv"))
folder = saved[-1].parent if saved else scan_originals()
if not (folder / "blind_queue.csv").exists():
    prepare_queue(folder)
print("원본 영상 판독 준비 완료")

### 1. 원본에서 허브 중심 찍기

아래 셀의 기본 모드는 `PAIRED`입니다. 같은 영상·같은 수동 동작 구간 안에서 최대 5프레임 차이가 나는 고신뢰 직접 검출과 짝을 이룹니다. 판독 순서는 섞여 있습니다. 대상이 0쌍이면 기본 판독은 완료입니다.

별도로 원본에서 허브를 확인하고 싶으면 `mode = SPOT`으로 바꾸세요. 이 모드는 무릎 통과부터 접촉까지의 중간에 가깝고 이전 39장에 없던 프레임을 고릅니다. 요청한 비교와 별도의 폴더에 저장합니다.

가까운 쪽 원판의 **허브 한가운데를 클릭**하면 즉시 저장됩니다. 확신이 없으면 **판독 불가로 저장**을 누르세요. **다음 미판독**을 누르면 이어서 진행할 수 있습니다.

In [ ]:
mode = PAIRED
editor(folder, mode)

### 2. 저장을 확인한 뒤 예측 열기

모든 프레임을 저장한 뒤에만 이 셀에서 점수와 좌표를 공개합니다. 다시 열어도 저장한 점은 유지됩니다. 세로 오차는 모델 중심 y에서 직접 찍은 허브 y를 뺀 값이며, 양수는 화면 아래 방향입니다. 판독 불가·출력 없음·정답 박스 미매칭은 허브 오차를 비워 둡니다.

In [ ]:
show_after_review(folder, mode)

### 선별 기준

- 원판 입력 640, 직접 검출 ≥0.10, 저신뢰 후보 하한 0.001, 직전 박스 폭의 0.5배 안에서 시간 연결합니다. 각 영상의 첫 프레임부터 추적합니다.
- 시간 연결 또는 최종 출력 누락을 후보로 기록합니다. 수동 원판 박스가 있는 장면에서는 최종 박스 IoU가 0.50 미만인 위치 실패도 따로 기록합니다. 수동 허브를 직접 확인하기 전에는 시간 연결이 맞는 원판을 복구했는지 확정하기 어렵습니다.
- 대조군은 같은 영상·같은 동작 구간 내 직접 검출 중 신뢰도 ≥0.50이며, 수동 박스가 있을 때는 IoU ≥0.50이어야 합니다. 5프레임 안에서 대조군을 재사용하지 않고 짝을 만듭니다. 짝이 없는 후보는 별도로 기록합니다.
- 원판이 없는 영상은 배경 오탐 확인용입니다. 수동 시점이 부족한 영상의 결과는 구간 미확인으로 남깁니다. 두 집단 모두 허브 판독에서 제외합니다.